# Black-Scholes Options Pricing
> **Project: Bloomberg Terminal Clone — Week 1**  
> Black-Scholes gives the *fair price* of a European option today,  
> given where the stock is, how volatile it is, and how much time is left.

---

## 0. Imports

In [ ]:
import numpy as np
from scipy.stats import norm          # N(x) — cumulative normal distribution
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['text.color']       = '#e6edf3'
plt.rcParams['axes.labelcolor']  = '#e6edf3'
plt.rcParams['xtick.color']      = '#8b949e'
plt.rcParams['ytick.color']      = '#8b949e'
plt.rcParams['grid.color']       = '#21262d'
plt.rcParams['grid.linewidth']   = 0.5

print("Libraries loaded ✓")

---
## 1. The Intuition — What Problem Are We Solving?

You want the right to **buy AAPL at \$150** three months from now.  
AAPL is trading at \$148 today. How much should that right cost?

**Black-Scholes says:** the stock price follows a random walk (geometric Brownian motion):

$$dS = \mu S \, dt + \sigma S \, dW_t$$

Which means at expiry, $S_T$ is **log-normally distributed**:

$$S_T = S_t \cdot e^{\left(r - \frac{1}{2}\sigma^2\right)(T-t) \;+\; \sigma\sqrt{T-t}\cdot Z}, \quad Z \sim \mathcal{N}(0,1)$$

The option price is the **discounted expected payoff** under the risk-neutral measure:

$$\boxed{\text{Price} = e^{-r(T-t)} \int_{-\infty}^{\infty} f\!\left(S_t \cdot e^{(r-\frac{1}{2}\sigma^2)(T-t)+\sigma\sqrt{T-t}\cdot z}\right) \cdot \frac{e^{-\frac{1}{2}z^2}}{\sqrt{2\pi}} \, dz}$$

> ⚠️ **Note from image:** Your formula is correct! Just remember the front term is $e^{-r(T-t)}$ (negative exponent) — we're *discounting* back to today.

When you solve this integral for a **Call** where $f(S_T) = \max(S_T - K,\, 0)$, it simplifies to the closed form below.

---
## 2. The Closed-Form Formula

$$\text{Call} = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$\text{Put}  = K e^{-rT} \cdot N(-d_2) - S \cdot N(-d_1)$$

where:

$$d_1 = \frac{\ln(S/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

**What each piece means:**
| Term | Meaning |
|------|---------|
| $S \cdot N(d_1)$ | Expected stock price × prob you exercise |
| $K e^{-rT} \cdot N(d_2)$ | Discounted strike × prob option expires ITM |
| $N(d_2)$ | Probability option expires in-the-money |
| $N(d_1)$ | The Delta of the call |

---
## 3. Core Black-Scholes Implementation

In [ ]:
def black_scholes(S, K, T_days, r, sigma, option_type='call'):
    """
    Black-Scholes European Option Pricer.

    Parameters
    ----------
    S           : float  — Current stock price          e.g. 150.0
    K           : float  — Strike price                 e.g. 155.0
    T_days      : float  — Days to expiry               e.g. 30
    r           : float  — Risk-free rate (decimal)     e.g. 0.05 = 5%
    sigma       : float  — Implied volatility (decimal) e.g. 0.25 = 25%
    option_type : str    — 'call' or 'put'

    Returns
    -------
    float — Fair price of the option
    """

    # --- Input validation ---
    if T_days <= 0:
        raise ValueError("T_days must be > 0. Option has expired.")
    if sigma <= 0:
        raise ValueError("sigma (volatility) must be > 0.")
    if S <= 0 or K <= 0:
        raise ValueError("Stock price and strike must be positive.")

    T = T_days / 365.0          # convert days → years

    # --- d1 and d2 ---
    # d1: how many std deviations the stock needs to move (drift-adjusted)
    # d2: pure probability term (d1 minus one std dev over the period)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # --- Option price ---
    # norm.cdf(x) = N(x) = cumulative standard normal distribution
    discount = np.exp(-r * T)   # e^(-rT) — present value factor

    if option_type.lower() == 'call':
        # Call = S·N(d1) - K·e^(-rT)·N(d2)
        price = S * norm.cdf(d1) - K * discount * norm.cdf(d2)

    elif option_type.lower() == 'put':
        # Put = K·e^(-rT)·N(-d2) - S·N(-d1)
        # N(-x) = 1 - N(x) — probability of the OPPOSITE outcome
        price = K * discount * norm.cdf(-d2) - S * norm.cdf(-d1)

    else:
        raise ValueError("option_type must be 'call' or 'put'")

    return round(price, 4)


# ---- Quick test ----
# AAPL-like example: stock at $150, strike $155, 30 days, 5% rate, 25% vol
call_price = black_scholes(S=150, K=155, T_days=30, r=0.05, sigma=0.25, option_type='call')
put_price  = black_scholes(S=150, K=155, T_days=30, r=0.05, sigma=0.25, option_type='put')

print(f"Call price : ${call_price}")
print(f"Put  price : ${put_price}")

# Put-Call Parity check: Call - Put = S - K·e^(-rT)
# If this holds, your formula is correct.
T = 30/365
parity = call_price - put_price
expected = 150 - 155 * np.exp(-0.05 * T)
print(f"\nPut-Call Parity check:")
print(f"  Call - Put       = {parity:.4f}")
print(f"  S - K·e^(-rT)   = {expected:.4f}")
print(f"  Match: {np.isclose(parity, expected, atol=1e-3)} ✓")

---
## 4. The Greeks — Sensitivities of the Option Price

In [ ]:
def greeks(S, K, T_days, r, sigma):
    """
    Compute all 5 Black-Scholes Greeks for a CALL option.

    Returns a dict with: delta, gamma, vega, theta, rho

    Greek    | Measures                        | Formula key
    ---------|---------------------------------|-------------------------
    Delta  Δ | dPrice/dS  (sensitivity to S)  | N(d1)
    Gamma  Γ | dΔ/dS      (curvature)          | N'(d1)/(S·σ·√T)
    Vega   ν | dPrice/dσ  (vol sensitivity)    | S·N'(d1)·√T / 100
    Theta  θ | dPrice/dt  (time decay/day)     | complex — see below
    Rho    ρ | dPrice/dr  (rate sensitivity)   | K·T·e^(-rT)·N(d2)/100
    """

    T = T_days / 365.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # N'(d1) = standard normal PDF at d1 — the "bell curve height"
    # This appears in every Greek formula — key building block
    pdf_d1   = norm.pdf(d1)
    discount = np.exp(-r * T)

    # --- DELTA ---
    # Call: N(d1) — always between 0 and 1
    # Put:  N(d1) - 1 — always between -1 and 0
    delta_call = norm.cdf(d1)
    delta_put  = delta_call - 1            # shortcut: call_delta - 1

    # --- GAMMA ---
    # Same for calls and puts. Always positive.
    # Highest when S ≈ K (at-the-money) and near expiry.
    gamma = pdf_d1 / (S * sigma * np.sqrt(T))

    # --- VEGA ---
    # Same for calls and puts. Always positive.
    # We divide by 100 → gives price change per 1% vol move
    vega = S * pdf_d1 * np.sqrt(T) / 100

    # --- THETA (call) ---
    # Always NEGATIVE — time passing hurts option buyers.
    # Divide by 365 to get per-day decay.
    theta_call = (
        -(S * pdf_d1 * sigma) / (2 * np.sqrt(T))      # vol component
        - r * K * discount * norm.cdf(d2)              # rate component
    ) / 365

    # --- RHO (call) ---
    # Positive for calls (rates up → calls more valuable)
    # Divide by 100 → gives price change per 1% rate move
    rho_call = K * T * discount * norm.cdf(d2) / 100

    return {
        'delta_call': round(delta_call, 4),
        'delta_put' : round(delta_put,  4),
        'gamma'     : round(gamma,      6),
        'vega'      : round(vega,       4),
        'theta'     : round(theta_call, 4),
        'rho'       : round(rho_call,   4),
    }


# ---- Test the Greeks ----
g = greeks(S=150, K=155, T_days=30, r=0.05, sigma=0.25)

print("Greeks for: S=150, K=155, T=30d, r=5%, σ=25%")
print("-" * 50)
print(f"Delta  Δ (call) : {g['delta_call']:>8}   → call moves ${g['delta_call']} per $1 stock move")
print(f"Delta  Δ (put)  : {g['delta_put']:>8}   → put  moves ${g['delta_put']} per $1 stock move")
print(f"Gamma  Γ        : {g['gamma']:>8}   → Delta changes by {g['gamma']} per $1 stock move")
print(f"Vega   ν        : {g['vega']:>8}   → price changes ${g['vega']} per 1% vol move")
print(f"Theta  θ        : {g['theta']:>8}   → loses ${abs(g['theta'])} per day just from time passing")
print(f"Rho    ρ        : {g['rho']:>8}   → price changes ${g['rho']} per 1% rate move")

---
## 5. IV Solver — Newton-Raphson

**The reverse problem:** You see a call trading at \$5.20 on the market.  
What volatility does the market *imply*? Black-Scholes has no inverse formula,  
so we iterate until our BS price matches the market price.

**Newton-Raphson step:**
$$\sigma_{n+1} = \sigma_n + \frac{\text{market price} - BS(\sigma_n)}{\text{Vega}(\sigma_n)}$$

In [ ]:
def solve_iv(market_price, S, K, T_days, r,
             option_type='call', tol=1e-6, max_iter=100):
    """
    Implied Volatility solver using Newton-Raphson.

    Given a market-observed option price, find the σ that BS
    would need to produce that exact price.

    Returns
    -------
    float : implied vol as a percentage (e.g. 24.5 means 24.5%)
    None  : if solver fails to converge (e.g. deep OTM near expiry)
    """

    # --- Sanity check: intrinsic value boundary ---
    # A call can never be worth less than max(S-K, 0)
    T = T_days / 365.0
    intrinsic = max(S - K, 0) if option_type == 'call' else max(K - S, 0)
    if market_price < intrinsic:
        print(f"  ⚠ Market price ${market_price} < intrinsic ${intrinsic}. Arbitrage or bad data.")
        return None

    sigma = 0.20    # initial guess: 20% vol (reasonable starting point)

    for i in range(max_iter):
        # Price the option at current sigma guess
        bs_price = black_scholes(S, K, T_days, r, sigma, option_type)

        # Compute Vega (we need the RAW vega, not per-1%)
        g = greeks(S, K, T_days, r, sigma)
        vega_raw = g['vega'] * 100          # undo the /100 we applied earlier

        diff = market_price - bs_price      # how far off are we?

        # Convergence check
        if abs(diff) < tol:
            return round(sigma * 100, 2)    # return as percentage

        # --- CRITICAL GUARD ---
        # Vega collapses to ~0 for deep OTM options near expiry.
        # Without this guard, we divide by zero and get garbage.
        if abs(vega_raw) < 1e-10:
            print(f"  ⚠ Vega ≈ 0 at iteration {i}. Deep OTM or near expiry. Cannot solve.")
            return None

        # Newton-Raphson step: move sigma in the direction that reduces error
        sigma = sigma + diff / vega_raw

        # Clamp sigma to realistic range [0.1%, 500%]
        # Negative sigma is mathematically impossible
        sigma = max(0.001, min(sigma, 5.0))

    print(f"  ⚠ Did not converge after {max_iter} iterations.")
    return None


# ---- Test the IV solver ----
# Step 1: Generate a BS price with known sigma
true_sigma  = 0.27                  # 27% — the "true" vol we want to recover
known_price = black_scholes(S=150, K=155, T_days=30, r=0.05, sigma=true_sigma)

# Step 2: Feed that price into IV solver — can it recover 27%?
solved_iv   = solve_iv(market_price=known_price, S=150, K=155, T_days=30, r=0.05)

print(f"True sigma  : {true_sigma*100:.2f}%")
print(f"BS price    : ${known_price}")
print(f"Solved IV   : {solved_iv}%")
print(f"Correct?    : {np.isclose(true_sigma*100, solved_iv, atol=0.01)} ✓")

# ---- Test the danger case ----
print("\n--- Testing danger case (deep OTM, 1 day to expiry) ---")
solve_iv(market_price=0.001, S=100, K=200, T_days=1, r=0.05)

---
## 6. Visualization — Greeks vs Stock Price

In [ ]:
# Parameters fixed — we vary S to see how Greeks change with stock price
K, T_days, r, sigma = 100, 45, 0.05, 0.25
S_range = np.linspace(60, 140, 200)

# Compute all Greeks across stock prices
calls, puts, deltas, gammas, vegas, thetas = [], [], [], [], [], []

for S in S_range:
    calls.append(black_scholes(S, K, T_days, r, sigma, 'call'))
    puts.append(black_scholes(S, K, T_days, r, sigma, 'put'))
    g = greeks(S, K, T_days, r, sigma)
    deltas.append(g['delta_call'])
    gammas.append(g['gamma'])
    vegas.append(g['vega'])
    thetas.append(g['theta'])

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

def style_ax(ax, title, ylabel):
    ax.set_title(title, color='#e6edf3', fontsize=11, pad=8)
    ax.set_xlabel('Stock Price S ($)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.axvline(K, color='#8b949e', lw=0.8, ls='--', alpha=0.6, label=f'Strike K={K}')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

# 1. Option prices
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(S_range, calls, color='#1D9E75', lw=1.8, label='Call')
ax1.plot(S_range, puts,  color='#993C1D', lw=1.8, label='Put')
style_ax(ax1, 'Option Price', 'Price ($)')

# 2. Delta
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(S_range, deltas, color='#378ADD', lw=1.8, label='Call Δ')
ax2.axhline(0.5, color='#8b949e', lw=0.6, ls=':', alpha=0.5)
style_ax(ax2, 'Delta Δ — Speed', 'Delta')

# 3. Gamma
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(S_range, gammas, color='#1D9E75', lw=1.8, label='Gamma Γ')
style_ax(ax3, 'Gamma Γ — Acceleration', 'Gamma')

# 4. Vega
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(S_range, vegas, color='#BA7517', lw=1.8, label='Vega ν')
style_ax(ax4, 'Vega ν — Vol Sensitivity (per 1%)', 'Vega ($)')

# 5. Theta
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(S_range, thetas, color='#993C1D', lw=1.8, label='Theta θ')
ax5.fill_between(S_range, thetas, 0, alpha=0.15, color='#993C1D')
style_ax(ax5, 'Theta θ — Daily Decay', 'Theta ($/day)')

# 6. Theta decay over time (ATM option)
ax6 = fig.add_subplot(gs[1, 2])
days_range = np.arange(1, 121)
theta_over_time = [greeks(100, 100, d, r, sigma)['theta'] for d in days_range]
ax6.plot(days_range, theta_over_time, color='#533AB7', lw=1.8, label='ATM θ over time')
ax6.set_xlabel('Days to Expiry', fontsize=9)
ax6.set_ylabel('Theta ($/day)', fontsize=9)
ax6.set_title('Theta Accelerates Near Expiry', color='#e6edf3', fontsize=11, pad=8)
ax6.invert_xaxis()      # time flows left to right → expiry on the right
ax6.grid(True, alpha=0.3)
ax6.legend(fontsize=8)
ax6.fill_between(days_range, theta_over_time, 0, alpha=0.15, color='#533AB7')

fig.suptitle('Black-Scholes Greeks   |   K=100, T=45d, r=5%, σ=25%',
             color='#e6edf3', fontsize=13, y=1.01)

plt.savefig('/home/claude/bs_greeks.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print("Plot saved ✓")

---
## 7. Putting It All Together — Full Options Chain Scanner
This is the function you'll call from your terminal backend.

In [ ]:
import pandas as pd

def price_options_chain(S, strikes, T_days, r, sigma):
    """
    Price a full options chain for a given stock.
    Returns a DataFrame — ready to display in your terminal.

    Parameters
    ----------
    S       : float       — Current stock price
    strikes : list/array  — List of strike prices
    T_days  : int         — Days to expiry
    r       : float       — Risk-free rate
    sigma   : float       — Implied volatility (flat — real world uses a surface)
    """
    rows = []
    for K in strikes:
        call_price = black_scholes(S, K, T_days, r, sigma, 'call')
        put_price  = black_scholes(S, K, T_days, r, sigma, 'put')
        g = greeks(S, K, T_days, r, sigma)

        # Moneyness tag
        if K < S * 0.97:
            moneyness = 'ITM'   # in-the-money
        elif K > S * 1.03:
            moneyness = 'OTM'   # out-of-the-money
        else:
            moneyness = 'ATM'   # at-the-money

        rows.append({
            'Strike'    : K,
            'Moneyness' : moneyness,
            'Call $'    : call_price,
            'Put $'     : put_price,
            'Delta Δ'   : g['delta_call'],
            'Gamma Γ'   : g['gamma'],
            'Vega ν'    : g['vega'],
            'Theta θ'   : g['theta'],
        })

    df = pd.DataFrame(rows).set_index('Strike')
    return df


# ---- Example: SPY-like chain ----
S = 150
strikes = np.arange(130, 172, 2)    # $130 to $170, every $2

chain = price_options_chain(
    S=S, strikes=strikes,
    T_days=30, r=0.05, sigma=0.25
)

print(f"Options chain for S=${S}  |  30 days  |  σ=25%\n")
pd.set_option('display.float_format', '{:.4f}'.format)
print(chain.to_string())
print(f"\nATM strike ≈ ${S} — Delta ≈ 0.50, Gamma at peak")

---
## 8. Quick Reference — Assumptions & Limitations

| Assumption | Reality | Impact on your terminal |
|---|---|---|
| Constant volatility σ | Vol changes constantly (smile/skew) | Use a vol surface (your Week 4) |
| Log-normal price moves | Fat tails, jumps exist | BS underprices tail risk |
| European options only | American options can be exercised early | Use binomial tree for American |
| Continuous trading | Markets have gaps, halts | Handle weekends, holidays |
| No dividends | Stocks pay dividends | Adjust S → S - PV(dividends) |

**Bottom line:** BS is wrong in every assumption. It's still the industry standard because  
it's fast, tractable, and gives a common language (IV) for quoting options.

> **Your Vol Surface project (Week 4) exists precisely because BS uses flat σ — but real markets show a "smile" where IV varies by strike and expiry.**